# IgarapIA no Google Colab

Antes de começar: **Ambiente de execução > Alterar o tipo de ambiente de execução > GPU (T4)**.

1. Descompacte `igarapia.zip` dentro do seu Google Drive, em `MeuDrive/igarapia`.
2. Coloque os vídeos dos igarapés em `MeuDrive/igarapia/dados/videos_brutos/`.
3. Rode as células na ordem. Tudo fica salvo no Drive (modelos, tabelas e vídeos).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/igarapia'
%cd $PROJ
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. Conferir a regra de contagem

In [ ]:
!python tests/test_contagem.py

## 1. Teste rápido com o modelo genérico (COCO)

Conta só garrafas e copos. Troque `teste.mp4` pelo nome de um vídeo curto seu.

In [ ]:
!python scripts/07_contar_video.py --video dados/videos_brutos/teste.mp4 --modelo yolo11n.pt --classes-coco bottle cup

## 2. Extrair quadros para anotação (2 por segundo)

In [ ]:
!python scripts/01_extrair_quadros.py --videos dados/videos_brutos --saida dados/quadros --passo 15

Agora anotem os quadros de `dados/quadros/` no CVAT ou Label Studio, com as classes
`garrafa_plastica`, `sacola_filme_plastico`, `isopor`, `lata_metal`, `outros_residuos`
(nessa ordem), exportem em formato YOLO e coloquem os `.txt` em `dados/rotulos/<video>/`.

## 3. Converter as bases públicas

Ajuste os caminhos para onde vocês baixaram cada base. Primeiro com `--so-listar`.

In [ ]:
!python scripts/02_converter_anotacoes.py --formato coco --fonte taco --so-listar \
    --anotacoes bases/TACO/data/annotations.json --imagens bases/TACO/data --saida dados/publicos/taco

In [ ]:
!python scripts/02_converter_anotacoes.py --formato coco --fonte taco \
    --anotacoes bases/TACO/data/annotations.json --imagens bases/TACO/data --saida dados/publicos/taco

## 4. Dividir por vídeo e montar os cenários E1, E2 e E3

In [ ]:
!python scripts/03_dividir_por_video.py --quadros dados/quadros --rotulos dados/rotulos --saida dados/local
!python scripts/04_montar_cenarios.py

## 5. Treinar (um cenário por célula; cada um pode levar de minutos a algumas horas)

In [ ]:
!python scripts/05_treinar.py --dados cenarios/E1.yaml --modelo yolo11n.pt --nome E1_yolo11n

In [ ]:
!python scripts/05_treinar.py --dados cenarios/E2.yaml --modelo yolo11n.pt --nome E2_yolo11n

In [ ]:
!python scripts/05_treinar.py --dados cenarios/E3.yaml --modelo yolo11n.pt --nome E3_yolo11n

## 6. Comparar os cenários no conjunto de teste

In [ ]:
!python scripts/06_avaliar_deteccao.py --dados cenarios/E3.yaml --nomes E1 E2 E3 \
    --modelos runs/E1_yolo11n/weights/best.pt runs/E2_yolo11n/weights/best.pt runs/E3_yolo11n/weights/best.pt

## 7. Contar resíduos em um vídeo com o modelo treinado

In [ ]:
!python scripts/07_contar_video.py --video dados/videos_brutos/mindu_01.mp4 \
    --modelo runs/E2_yolo11n/weights/best.pt --salvar-mot --inicio "2026-11-10 08:30:00"

## 8. Comparar com a contagem manual (crie `dados/contagem_manual.csv` antes)

In [ ]:
!python scripts/08_avaliar_contagem.py --manual dados/contagem_manual.csv --resultados resultados

## 9. MOTA e IDF1 em um trecho anotado com identificadores (exportado do CVAT em MOT 1.1)

In [ ]:
!python scripts/09_avaliar_rastreamento.py --gt anotacoes_mot/mindu_01/gt/gt.txt --pred resultados/mindu_01/rastreamento_mot.txt